# 01 - EDA · DataCo Smart Supply Chain

Análisis Exploratorio de Datos sobre el dataset **DataCo Smart Supply Chain Dataset** de Kaggle.

🔗 [Dataset en Kaggle](https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis)

## Objetivos del notebook

1. Cargar correctamente el CSV (encoding `latin-1`).
2. Inspeccionar `shape`, `info`, `describe`.
3. Analizar valores nulos y duplicados.
4. Estudiar la distribución de la variable objetivo `Late_delivery_risk`.
5. Visualizar histogramas y correlaciones de variables numéricas.
6. Extraer **insights iniciales** para el modelado.
7. Ejecutar el pipeline de limpieza y guardar el dataset limpio en `data/processed/`.

## Cómo obtener el dataset

1. Descargar desde Kaggle: `DataCoSupplyChainDataset.csv`.
2. Colocarlo en `data/raw/`.
3. Ejecutar las celdas de este notebook.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Permitir importar módulos desde src/
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    cargar_dataco,
    normalizar_nombres_columnas,
    reporte_nulos,
    convertir_fechas,
    eliminar_duplicados,
    eliminar_columnas_totalmente_nulas,
    eliminar_columnas_sensibles,
    limpiar_dataco,
)
from src.utils import configurar_logger

configurar_logger('eda_dataco')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Carga del dataset

El CSV de DataCo usa encoding `latin-1`. La función `cargar_dataco` intenta varios encodings comunes de manera robusta.

In [ ]:
df = cargar_dataco('DataCoSupplyChainDataset.csv')
df.head()

## 2. Shape, info y describe

In [ ]:
print(f'Filas:    {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]:,}')
print(f'Memoria:  {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

In [ ]:
df.info(verbose=True, show_counts=True)

In [ ]:
df.describe(include=[np.number]).T

In [ ]:
df.describe(include=['object']).T

## 3. Normalización de nombres de columnas

In [ ]:
df = normalizar_nombres_columnas(df)
df.columns.tolist()

## 4. Análisis de valores nulos

In [ ]:
nulos = reporte_nulos(df, umbral_pct=0.0)
nulos.head(20)

In [ ]:
# Visualización de las 15 columnas con más nulos
if not nulos.empty:
    top_nulos = nulos.head(15)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=top_nulos, x='pct_nulos', y='columna', ax=ax, color='#cc4c4c')
    ax.set_title('Top 15 columnas con mayor porcentaje de nulos')
    ax.set_xlabel('% nulos')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print('No hay columnas con nulos.')

## 5. Conversión de fechas y duplicados

In [ ]:
df = convertir_fechas(df)
df[['order_date_dateorders', 'shipping_date_dateorders']].dtypes

In [ ]:
print(f"Rango de fechas (orden):    {df['order_date_dateorders'].min()}  ->  {df['order_date_dateorders'].max()}")
print(f"Rango de fechas (envío):   {df['shipping_date_dateorders'].min()}  ->  {df['shipping_date_dateorders'].max()}")

In [ ]:
df = eliminar_duplicados(df)
df.shape

## 6. Distribución de la variable objetivo: `late_delivery_risk`

Esta es la variable que alimentará el clasificador de retrasos en `src/train_model.py`.

In [ ]:
target = 'late_delivery_risk'
distribucion = df[target].value_counts(normalize=True).sort_index()
print('Distribución de la variable objetivo:')
print(distribucion.apply(lambda x: f'{x:.2%}'))
print(f'\nTotal de registros positivos (1): {df[target].sum():,}')
print(f'Total de registros negativos (0): {(df[target]==0).sum():,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x=target, ax=axes[0], palette=['#4c8acc', '#cc4c4c'])
axes[0].set_title('Conteo de Late_delivery_risk')
axes[0].set_xlabel('Riesgo de retraso (0 = no, 1 = sí)')

axes[1].pie(
    distribucion.values,
    labels=['A tiempo', 'Retraso'],
    autopct='%1.1f%%',
    colors=['#4c8acc', '#cc4c4c'],
    startangle=90,
)
axes[1].set_title('Proporción de retrasos')
plt.tight_layout()
plt.show()

### Relación entre `delivery_status` y `late_delivery_risk`

In [ ]:
if 'delivery_status' in df.columns:
    tabla = pd.crosstab(df['delivery_status'], df[target], normalize='index') * 100
    print('% de retraso por estado de entrega:')
    print(tabla.round(2))

## 7. Histogramas de variables numéricas clave

In [ ]:
columnas_clave = [
    'days_for_shipping_real',
    'days_for_shipment_scheduled',
    'sales',
    'order_item_quantity',
    'order_item_discount',
    'order_item_discount_rate',
    'order_item_profit_ratio',
    'product_price',
]
columnas_clave = [c for c in columnas_clave if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flat, columnas_clave):
    df[col].hist(bins=40, ax=ax, color='#4c8acc', edgecolor='white')
    ax.set_title(col, fontsize=10)
    ax.grid(False)
for ax in axes.flat[len(columnas_clave):]:
    ax.axis('off')
plt.suptitle('Histogramas de variables numéricas clave', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### Días de envío real vs. programado

In [ ]:
if {'days_for_shipping_real', 'days_for_shipment_scheduled'}.issubset(df.columns):
    fig, ax = plt.subplots(figsize=(10, 5))
    df['days_for_shipping_real'].hist(bins=20, alpha=0.6, label='Real', ax=ax, color='#cc4c4c')
    df['days_for_shipment_scheduled'].hist(bins=20, alpha=0.6, label='Programado', ax=ax, color='#4c8acc')
    ax.set_xlabel('Días de envío')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Días reales vs. programados de envío')
    ax.legend()
    plt.show()
    
    diferencia = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']
    print(f'Diferencia promedio (real - programado): {diferencia.mean():.2f} días')
    print(f'% órdenes con días reales > programados: {(diferencia > 0).mean():.2%}')

## 8. Matriz de correlaciones (numéricas)

In [ ]:
df_num = df.select_dtypes(include=[np.number])
# Eliminamos IDs porque sesgan la correlación
cols_id = [c for c in df_num.columns if c.endswith('_id') or c == 'product_card_id']
df_num = df_num.drop(columns=cols_id, errors='ignore')

corr = df_num.corr()
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    corr,
    cmap='RdBu_r',
    center=0,
    annot=False,
    square=True,
    cbar_kws={'shrink': 0.8},
    ax=ax,
)
ax.set_title('Matriz de correlaciones · variables numéricas (sin IDs)')
plt.tight_layout()
plt.show()

### Top variables correlacionadas con la variable objetivo

In [ ]:
if target in corr.columns:
    corr_target = (
        corr[target]
        .drop(target)
        .abs()
        .sort_values(ascending=False)
        .head(15)
    )
    print('Top 15 variables más correlacionadas con late_delivery_risk:')
    print(corr_target.round(3))

## 9. Análisis inicial: retrasos por segmento

Exploramos cómo varía la tasa de retraso por **modo de envío**, **mercado**, **región** y **segmento de cliente**.

In [ ]:
def tasa_retraso_por(columna):
    if columna not in df.columns:
        return None
    agg = (
        df.groupby(columna)[target]
        .agg(['count', 'mean'])
        .rename(columns={'count': 'ordenes', 'mean': 'tasa_retraso'})
        .sort_values('tasa_retraso', ascending=False)
    )
    return agg

for col in ['shipping_mode', 'market', 'order_region', 'customer_segment']:
    resumen = tasa_retraso_por(col)
    if resumen is not None:
        print(f'\n--- Tasa de retraso por {col} ---')
        print(resumen.head(10))

In [ ]:
if 'shipping_mode' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    (
        df.groupby('shipping_mode')[target].mean()
        .sort_values(ascending=False)
        .plot(kind='barh', ax=ax, color='#4c8acc')
    )
    ax.set_title('Tasa de retraso por modo de envío')
    ax.set_xlabel('% de retrasos')
    plt.tight_layout()
    plt.show()

## 10. Conclusiones preliminares

_Completar con los hallazgos observados al ejecutar las celdas_:

- **Volumen de datos**: ~180.000 órdenes, ~53 columnas originales.
- **Calidad**: algunas columnas (`product_description`, `order_zipcode`) suelen tener altísimo % de nulos → candidatas a eliminación.
- **Variable objetivo**: `late_delivery_risk` suele estar entre 50-55% positiva → dataset bastante balanceado.
- **Variables relacionadas**: `days_for_shipping_real`, `days_for_shipment_scheduled`, `shipping_mode` y `delivery_status` aportan mucha señal.
- **Segmentación**: el modo de envío `Same Day` y `First Class` suele concentrar las mayores tasas de retraso.
- **Próximos pasos**:
    1. Ejecutar `limpiar_dataco()` para generar `data/processed/dataco_clean.parquet`.
    2. Pasar a `02_feature_engineering.ipynb` para construir features temporales y económicas.
    3. Entrenar el modelo XGBoost en `03_training_mlflow.ipynb`.

## 11. Ejecutar pipeline de limpieza y guardar

Aplica todo el pipeline orquestado y persiste el resultado en `data/processed/dataco_clean.parquet`.

In [ ]:
df_limpio = limpiar_dataco(
    nombre_archivo='DataCoSupplyChainDataset.csv',
    guardar=True,
    nombre_salida='dataco_clean.parquet',
    imputar=False,
)
df_limpio.shape

In [ ]:
df_limpio.head()